<a href="https://colab.research.google.com/github/jazaineam1/BigData2026/blob/main/Cuadernos/9_Databricks_Serverless_Completo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

> ⚠️ **Plataforma recomendada: Databricks con Serverless Notebook Compute o compute administrado moderno. Este notebook NO esta disenado para Databricks Community Edition clasico.**

# Databricks Serverless: la plataforma de datos moderna

## Universidad Central
> ### Facultad de Ingeniería y Ciencias Básicas
> ### Maestría en Analítica de Datos -- Big Data

![Universidad Central](https://www.ucentral.edu.co/themes/ucentral/img/template/Universidad%20Central.png)

> **Sesión 9** · 2026

## Proposito pedagogico

Esta sesion es la continuacion natural despues de Pandas, Dask y la introduccion a Spark
de la Sesion 8. El foco aqui es la **plataforma Databricks actual**, que en 2025-2026
ya no es solo "un cluster con Spark": es un ecosistema completo de compute serverless,
catalogos unificados, motores vectorizados y pipelines declarativos.

### Al final de esta sesion deberias poder:

1. Detectar y entender el entorno Databricks en el que estas corriendo (Classic vs Serverless).
2. Usar Unity Catalog correctamente con nombres de 3 partes (`catalog.schema.table`).
3. Entender por que el plan de Catalyst importa tanto como el resultado final.
4. Crear tablas con Liquid Clustering y entender cuando usar OPTIMIZE.
5. Aplicar MERGE, Time Travel y VACUUM en Delta Lake.
6. Explicar con criterios claros cuando elegir Pandas, Dask o Spark.
7. Leer el codigo de un pipeline DLT y entender su arquitectura medallion.

## Contenido

- 0. Que es Databricks Serverless y como cambio la plataforma (2025-2026)
- 1. Spark Connect: el nuevo modelo cliente-servidor
- 2. Unity Catalog: 3 niveles y Volumes
- 3. Spark en profundidad: lazy evaluation, Catalyst y plan fisico
- 4. Photon y Liquid Clustering: optimizacion automatica
- 5. Transformaciones completas de Spark con samples.nyctaxi.trips
- 6. Por que Spark sobre Pandas -- y cuando no
- 7. Por que Spark sobre Dask -- y cuando no
- 8. Delta Lake avanzado: MERGE, Time Travel, VACUUM
- 9. Delta Live Tables (DLT): el patron medallion
- 10. Taller: pipeline end-to-end serverless

---
# Sección 0 -- Que es Databricks Serverless y como cambio la plataforma

## La plataforma en 2025-2026

Databricks ha evolucionado. Si conoces el material de sesiones anteriores donde se creaba
un cluster `Single Node` manualmente y se esperaban 5-10 minutos, ese flujo ya no es el default.

La nueva plataforma gira alrededor de **compute serverless**: infraestructura que arranca en
segundos, se escala automaticamente y no requiere que el estudiante o el ingeniero configure
nodos, runtime ni recursos.

### Comparacion: Classic vs Serverless

| Aspecto | Classic Cluster | Serverless Compute |
|---|---|---|
| Tiempo de arranque | 5-10 min | ~10 segundos |
| Gestion de infraestructura | Manual (tipo, nodos, runtime) | Automatica |
| `sparkContext` disponible | Si | No siempre |
| DBFS root `/dbfs/` recomendado | Si (legacy) | No — usar Unity Catalog |
| Spark UI clasica | Si | Query Profile |
| Costo | Por cluster-hora fija | Por DBU consumida |
| Actualizacion de runtime | Manual | Automatica |

### Componentes clave del ecosistema actual

```
+----------------------------------------------------------+
|                    DATABRICKS PLATFORM                    |
|                                                          |
|  [Serverless Notebook]  [SQL Warehouse]  [DLT Pipeline]  |
|          |                    |                |          |
|          +--------+-----------+--------+-------+          |
|                   |                    |                  |
|           [Spark Connect]      [Unity Catalog]            |
|           (gRPC client)        catalog.schema.table       |
|                   |                    |                  |
|           [Spark Engine]       [Delta Lake / Volumes]     |
|           + Photon (C++)       + Liquid Clustering        |
|           + Catalyst           + Predictive Optimization  |
+----------------------------------------------------------+
```

### Lo que cambia para el codigo

- Todas las tablas se referencian como `catalog.schema.table` (Unity Catalog).
- Las rutas legacy `/dbfs/FileStore/...` ya no son la forma recomendada.
- `spark.sparkContext` puede no estar disponible en Serverless (Spark Connect).
- Los paquetes se instalan con `%pip`, no con `%sh pip install`.
- La Spark UI clasica puede reemplazarse por **Query Profile** en el notebook.

In [ ]:
# Deteccion completa del entorno Databricks 2025
import sys

print(f"Python: {sys.version}")
print(f"Spark:  {spark.version}")

IS_SERVERLESS      = False
HAS_SPARK_CONTEXT  = False
HAS_UNITY_CATALOG  = False

try:
    _ = spark.sparkContext.master
    HAS_SPARK_CONTEXT = True
    print("Compute: clasico (sparkContext disponible)")
except Exception:
    IS_SERVERLESS = True
    print("Compute: serverless / Spark Connect (sparkContext no disponible)")

try:
    current_cat = spark.sql("SELECT current_catalog()").first()[0]
    HAS_UNITY_CATALOG = current_cat not in ("", None, "hive_metastore")
    print(f"Catalogo activo  : {current_cat}")
    print(f"Unity Catalog    : {'si' if HAS_UNITY_CATALOG else 'no (hive_metastore legacy)'}")
except Exception as exc:
    print(f"Catalogo         : no detectable ({exc})")

try:
    photon = spark.conf.get("spark.databricks.photon.enabled", "false")
    print(f"Photon           : {photon}")
except Exception:
    print("Photon           : no detectable en este entorno")

print(f"\nResumen: IS_SERVERLESS={IS_SERVERLESS}, HAS_UNITY_CATALOG={HAS_UNITY_CATALOG}")

In [ ]:
# Instalar dependencias para comparaciones de esta sesion
# REGLA: siempre %pip en Databricks (no %sh pip install)
# %sh pip puede quedar instalado en el sistema pero no en el interprete activo del notebook
%pip install "dask[dataframe]>=2024.1" pyarrow -q

## Por que `%pip` y no `%sh pip install`

En Databricks, el interprete Python del notebook puede estar separado del sistema de archivos del nodo.

- `%sh pip install foo` instala en el sistema, pero el interprete del notebook puede no verlo.
- `%pip install foo` instala directamente en el entorno Python del notebook y, si es necesario, reinicia el interprete automaticamente.

**Regla practica**: en cualquier entorno Databricks moderno, usa siempre `%pip`.

---
# Sección 1 -- Spark Connect: el nuevo modelo cliente-servidor

## El modelo antiguo vs el nuevo

### Antes (Classic cluster):
```
+------------------------+
|  Python Process        |
|  +-----------------+   |
|  | SparkContext    |   |  <- Python y JVM en el mismo proceso
|  | (JVM embedded)  |   |
|  +-----------------+   |
+------------------------+
```

### Ahora (Spark Connect / Serverless):
```
+-------------------+          gRPC          +----------------------+
|  Python Client    |  <------------------> |  Spark Server        |
|  (tu notebook)    |                        |  (JVM remoto)        |
|  DataFrame API    |                        |  Catalyst optimizer  |
|  SQL              |                        |  Tungsten            |
+-------------------+                        +----------------------+
```

La consecuencia practica es que `sparkContext`, que vivia incrustado en el proceso Python,
ya no esta disponible directamente. El cliente solo ve la `SparkSession`.

### Que cambia para el codigo

| API | Spark Connect | Classic |
|---|---|---|
| `spark.sql(...)` | Funciona | Funciona |
| `spark.read.table(...)` | Funciona | Funciona |
| DataFrame API completa | Funciona | Funciona |
| `spark.sparkContext.master` | Puede fallar | Funciona |
| `spark.sparkContext.uiWebUrl` | Puede fallar | Funciona |
| `spark.sparkContext.parallelize(...)` | Puede fallar | Funciona |
| `rdd.getNumPartitions()` | Puede fallar | Funciona |

**Regla de oro**: en Serverless, usa `spark.createDataFrame(...)` en lugar de
`spark.sparkContext.parallelize(...)`, y evita las APIs de bajo nivel del RDD.

In [ ]:
# Demostracion: que funciona y que no en Spark Connect
from pyspark.sql import functions as F

# --- Lo que siempre funciona ---
df_test = spark.range(10)
print("spark.range(10).count() ->", df_test.count())

resultado = spark.sql("SELECT 1 + 1 AS suma")
print("spark.sql('SELECT 1+1') ->", resultado.first()["suma"])

# --- sparkContext: puede o no estar disponible ---
if globals().get("HAS_SPARK_CONTEXT", False):
    print("sparkContext.master:", spark.sparkContext.master)
    print("sparkContext.uiWebUrl:", spark.sparkContext.uiWebUrl)
else:
    print("sparkContext: no disponible en este entorno (Spark Connect / serverless)")

# --- Alternativa correcta a parallelize ---
# NO usar: spark.sparkContext.parallelize([1, 2, 3])
# SI usar:
local_data = spark.createDataFrame([(1,), (2,), (3,)], ["valor"])
local_data.show()

---
# Sección 2 -- Unity Catalog: 3 niveles y Volumes

## La jerarquia de 3 niveles

Unity Catalog organiza todos los objetos de datos en una jerarquia de tres niveles:

```
catalog
  └── schema  (tambien llamado "database" en Spark clasico)
        └── table | view | function | volume
```

Ejemplos reales:

| Ruta completa | Significado |
|---|---|
| `samples.nyctaxi.trips` | Tabla publica de demo de Databricks |
| `workspace.default.mi_tabla` | Tu tabla en el schema default del workspace |
| `main.bronze.raw_events` | Tabla en capa bronze de un pipeline medallion |
| `/Volumes/main/bronze/raw_files/data.parquet` | Archivo en un Volume de Unity Catalog |

### Por que importa Unity Catalog

- **Gobernanza centralizada**: permisos a nivel de columna, no solo de tabla.
- **Linaje de datos**: Databricks registra de donde viene cada tabla y que la usa.
- **Audit log**: registro completo de accesos y modificaciones.
- **Reemplaza `hive_metastore`**: el catalogo legacy que usaban los clusters clasicos.

### Volumes: el reemplazo de DBFS root

En Serverless, la ruta `/dbfs/FileStore/...` ya no es recomendada.
Los **Volumes** son el lugar correcto para guardar archivos dentro de Unity Catalog:

```
/Volumes/<catalog>/<schema>/<volume>/<path>/<archivo>
```

Ejemplo: `/Volumes/main/bronze/raw_files/taxi_2024.parquet`

In [ ]:
# Explorar la jerarquia de Unity Catalog
print("=== Catalogos disponibles ===")
spark.sql("SHOW CATALOGS").show(truncate=False)

print("=== Catalog y schema actuales ===")
current_cat    = spark.sql("SELECT current_catalog()").first()[0]
current_schema = spark.sql("SELECT current_schema()").first()[0]
print(f"  Catalog : {current_cat}")
print(f"  Schema  : {current_schema}")

print("\n=== Tablas en samples.nyctaxi ===")
spark.sql("SHOW TABLES IN samples.nyctaxi").show(truncate=False)

In [ ]:
# Acceso correcto a una tabla con nombre de 3 partes
TAXI_TABLE = "samples.nyctaxi.trips"
sdf = spark.read.table(TAXI_TABLE)

print(f"Tabla: {TAXI_TABLE}")
print(f"Filas: {sdf.count():,}")
print(f"Columnas: {len(sdf.columns)}")
sdf.printSchema()

In [ ]:
# Volumes: almacenamiento de archivos en Unity Catalog
# En Serverless los Volumes reemplazan DBFS root como lugar de archivos

try:
    spark.sql("SHOW VOLUMES IN samples.nyctaxi").show(truncate=False)
except Exception as exc:
    print(f"No hay Volumes en samples.nyctaxi: {exc}")

print()
print("Ruta de un Volume en produccion:")
print("  /Volumes/<catalog>/<schema>/<volume>/<archivo>")
print("  Ejemplo: /Volumes/main/bronze/raw_files/taxi_2024.parquet")
print()
print("Para crear un Volume en tu schema:")
print("  CREATE VOLUME IF NOT EXISTS workspace.default.mis_archivos")

---
# Sección 3 -- Spark en profundidad: lazy evaluation, Catalyst y plan fisico

## El ciclo de vida de una query en Spark

Cuando escribes `sdf.filter(...).groupBy(...).agg(...)`, Spark **no ejecuta nada todavia**.
Construye un plan que pasa por varias etapas de optimizacion antes de tocar un solo dato:

```
Codigo Python
     |
     v
Unresolved Logical Plan
(los nombres de columnas aun no se validaron)
     |
     v
Resolved Logical Plan
(el Catalog valido que las columnas existen)
     |
     v
Optimized Logical Plan  <-- aqui actua CATALYST
(filtros adelantados, columnas innecesarias eliminadas,
 joins reordenados, predicados combinados...)
     |
     v
Physical Plans (varios candidatos)
     |
     v
Selected Physical Plan (el mas barato segun el cost model)
     |
     v
Ejecucion (Jobs, Stages, Tasks)
```

### La diferencia con Pandas

En Pandas, el codigo Python se ejecuta inmediatamente, linea a linea.
En Spark, el codigo Python describe un plan que se optimiza y ejecuta de forma diferida.

Eso es lo que se llama **lazy evaluation**: las transformaciones no ejecutan; solo las **acciones** disparan el trabajo.

Acciones comunes: `count()`, `show()`, `collect()`, `toPandas()`, `write.save(...)`.

In [ ]:
# Lazy evaluation: demostrar que las transformaciones no ejecutan
import time
from pyspark.sql import functions as F

t0 = time.perf_counter()
pipeline = (
    sdf
    .filter(F.col("fare_amount") > 0)
    .filter(F.col("trip_distance") > 0.1)
    .withColumn("pickup_hour", F.hour("tpep_pickup_datetime"))
    .withColumn(
        "tip_pct",
        F.when(F.col("fare_amount") > 0,
               F.col("tip_amount") / F.col("fare_amount") * 100)
         .otherwise(F.lit(0.0))
    )
    .withColumn(
        "categoria_viaje",
        F.when(F.col("trip_distance") < 1,  "micro")
         .when(F.col("trip_distance") < 3,  "corto")
         .when(F.col("trip_distance") < 10, "medio")
         .otherwise("largo")
    )
    .withColumn(
        "duracion_min",
        (F.unix_timestamp("tpep_dropoff_datetime") -
         F.unix_timestamp("tpep_pickup_datetime")) / 60
    )
)
t_plan = time.perf_counter() - t0

print(f"Tiempo en construir el plan: {t_plan*1000:.2f} ms")
print("El DataFrame 'pipeline' es solo un plan — ningun dato se proceso todavia.")
print(pipeline)

In [ ]:
# explain() en 3 modos para entender el plan
print("=== PLAN SIMPLE (solo fisico) ===")
pipeline.explain(False)

print("\n=== PLAN FORMATTED (mas legible) ===")
pipeline.explain("formatted")

## Como leer el plan fisico

| Operador en el plan | Que significa |
|---|---|
| `Project` | Seleccion de columnas (`select`, `withColumn`) |
| `Filter` | Predicado (`filter`, `where`) |
| `Exchange` | **Shuffle** — redistribucion de datos entre particiones. Caro. |
| `HashAggregate` | Agregacion local (antes del shuffle) |
| `Sort` | Ordenamiento |
| `BroadcastHashJoin` | Join donde una tabla es pequena y se replica. Eficiente. |
| `SortMergeJoin` | Join entre tablas grandes. Implica shuffle en ambos lados. |
| `PushedFilters` | Filtros empujados al lector de archivos (predicate pushdown) |

**Regla de deteccion rapida**: si ves `Exchange` en el plan, hay un shuffle. Preguntate si es inevitable.

In [ ]:
# Predicate pushdown: Catalyst empuja filtros al nivel del lector
# En tablas Delta con estadisticas, esto puede saltar bloques enteros de archivos

plan_con_filtro = (
    spark.read.table("samples.nyctaxi.trips")
    .filter(F.col("fare_amount").between(10, 50))
    .filter(F.col("trip_distance") > 1)
    .select("fare_amount", "trip_distance", "tip_amount")
)

print("Plan con predicate pushdown:")
plan_con_filtro.explain("formatted")

t0 = time.perf_counter()
n = plan_con_filtro.count()
print(f"\nFilas resultantes: {n:,} | Tiempo: {time.perf_counter()-t0:.2f}s")

In [ ]:
# Cache: materializar un DataFrame intermedio para reutilizarlo
# Util cuando el mismo DataFrame se usa en multiples acciones

base = (
    sdf
    .filter(F.col("fare_amount") > 0)
    .withColumn("pickup_hour", F.hour("tpep_pickup_datetime"))
    .withColumn("tip_pct", F.col("tip_amount") / F.col("fare_amount") * 100)
    .filter(F.col("trip_distance") > 0)
)

# Sin cache: cada accion re-lee y re-procesa desde el origen
t0 = time.perf_counter()
n1 = base.count()
t1 = time.perf_counter() - t0

t0 = time.perf_counter()
_ = base.filter(F.col("pickup_hour") == 8).count()
t2 = time.perf_counter() - t0

print(f"Sin cache  — accion 1: {t1:.2f}s | accion 2: {t2:.2f}s")

# Con cache: la primera accion materializa en memoria
base.cache()

t0 = time.perf_counter()
n1c = base.count()
t1c = time.perf_counter() - t0

t0 = time.perf_counter()
_ = base.filter(F.col("pickup_hour") == 8).count()
t2c = time.perf_counter() - t0

print(f"Con cache  — accion 1: {t1c:.2f}s | accion 2: {t2c:.2f}s")
if t2 > 0:
    print(f"Speedup segunda accion: {t2/max(t2c, 0.001):.1f}x")

base.unpersist()
print("Cache liberado.")

---
# Sección 4 -- Photon y Liquid Clustering: optimizacion automatica

## Photon: el motor vectorizado de Databricks

**Photon** es un motor de ejecucion nativo (escrito en C++) que Databricks usa en lugar
del motor JVM clasico de Spark para muchas operaciones.

### Como funciona

El motor JVM de Spark procesa datos fila por fila (row-at-a-time).
Photon procesa datos **columna por columna, en bloques vectorizados**, lo que permite:
- Usar instrucciones SIMD del CPU moderno.
- Reducir el overhead de la JVM.
- Acelerar especialmente: scans de columnas numericas, agregaciones, joins, filtros.

### Lo importante para el estudiante

**No necesitas cambiar tu codigo.** Photon se activa automaticamente en Databricks Serverless
cuando la operacion es compatible. El mismo `groupBy(...).agg(...)` que escribes en PySpark
puede ejecutarse con Photon sin ningun cambio.

Puedes verificar si Photon se uso revisando el **Query Profile** del notebook.

## Liquid Clustering: el fin de PARTITION BY + ZORDER

### El problema con el particionamiento clasico

En el mundo Spark/Delta clasico, se usaba:
```sql
CREATE TABLE mi_tabla
PARTITIONED BY (anio, mes)
AS SELECT ...
```
Y luego se optimizaba con:
```sql
OPTIMIZE mi_tabla ZORDER BY (ciudad, tipo_viaje)
```

El problema: habia que decidir las columnas de particion **de antemano**, antes de saber
exactamente como se consultaria la tabla. Si los patrones de consulta cambiaban, habia que
reorganizar la tabla.

### La solucion: Liquid Clustering

Con **Liquid Clustering**, la sintaxis es:
```sql
CREATE TABLE mi_tabla
CLUSTER BY (columna1, columna2)
AS SELECT ...
```

Ventajas:
- No hay que elegir el tamano de particion.
- El clustering se aplica incremental y automaticamente con `OPTIMIZE`.
- Se puede cambiar la estrategia de clustering sin reescribir la tabla completa.
- Databricks puede aplicar el clustering automaticamente con **Predictive Optimization**.

In [ ]:
# Crear tabla con Liquid Clustering
CATALOG = spark.sql("SELECT current_catalog()").first()[0]
SCHEMA  = spark.sql("SELECT current_schema()").first()[0]
LC_TABLE = f"{CATALOG}.{SCHEMA}.taxi_liquid_sesion9"

print(f"Creando tabla con Liquid Clustering: {LC_TABLE}")

spark.sql(f'''
    CREATE OR REPLACE TABLE {LC_TABLE}
    CLUSTER BY (tpep_pickup_datetime, fare_amount)
    AS
    SELECT *
    FROM samples.nyctaxi.trips
    WHERE fare_amount > 0
''')

spark.sql(f"DESCRIBE DETAIL {LC_TABLE}").select(
    "format", "clusteringColumns", "numFiles", "sizeInBytes"
).show(truncate=False)

In [ ]:
# OPTIMIZE aplica el clustering fisicamente en los archivos
print("Aplicando OPTIMIZE...")
spark.sql(f"OPTIMIZE {LC_TABLE}")

spark.sql(f"DESCRIBE HISTORY {LC_TABLE}").select(
    "version", "timestamp", "operation", "operationParameters"
).show(5, truncate=False)

print("\nDespues del OPTIMIZE, los archivos estan organizados segun las columnas de clustering.")
print("Las queries que filtran por tpep_pickup_datetime o fare_amount pueden saltar archivos irrelevantes.")

## Predictive Optimization

En workspaces con **Predictive Optimization** habilitado (disponible en cuentas Premium),
Databricks ejecuta `OPTIMIZE` y `VACUUM` automaticamente basandose en:
- El historial de uso de la tabla.
- La frecuencia de escrituras.
- Los patrones de lectura.

Esto reduce la deuda operacional: no necesitas programar jobs de mantenimiento manualmente.

Para verificar si esta activo: Settings → Delta → Predictive Optimization en tu workspace.

---
# Sección 5 -- Transformaciones completas de Spark con samples.nyctaxi.trips

## El tour completo de la API PySpark

Esta seccion cubre las transformaciones mas importantes de Spark DataFrame API,
todas sobre el mismo dataset. La idea es que puedas ver en un solo lugar como se usan,
que operaciones causan shuffle y como leer el plan resultante.

### Clasificacion de transformaciones

| Tipo | Operaciones | Shuffle? |
|---|---|---|
| **Narrow** | `select`, `filter`, `withColumn`, `drop`, `alias` | No |
| **Wide** | `groupBy`, `join` (no broadcast), `distinct`, `orderBy` | Si |
| **Especiales** | `cache`, `repartition`, `coalesce` | Depende |

Una transformacion **narrow** puede procesarse dentro de cada particion sin mover datos.
Una transformacion **wide** necesita redistribuir datos entre particiones (**shuffle**).

In [ ]:
# Setup del dataset y variables globales de sesion
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import StringType
import time

# Cargar el dataset completo
sdf_full = spark.read.table("samples.nyctaxi.trips")
TOTAL = sdf_full.count()
print(f"Dataset completo: {TOTAL:,} filas | {len(sdf_full.columns)} columnas")

In [ ]:
# --- select, alias, withColumn, when/otherwise ---
enriquecido = (
    sdf_full
    .select(
        F.col("tpep_pickup_datetime").alias("pickup_ts"),
        F.col("tpep_dropoff_datetime").alias("dropoff_ts"),
        F.col("fare_amount"),
        F.col("tip_amount"),
        F.col("trip_distance"),
        F.col("passenger_count"),
        F.col("pickup_zip"),
    )
    .filter(F.col("fare_amount").between(1, 200))
    .filter(F.col("trip_distance") > 0)
    .withColumn("pickup_hour", F.hour("pickup_ts"))
    .withColumn("pickup_dow",  F.dayofweek("pickup_ts"))
    .withColumn(
        "duracion_min",
        (F.unix_timestamp("dropoff_ts") - F.unix_timestamp("pickup_ts")) / 60
    )
    .withColumn(
        "velocidad_mph",
        F.when(F.col("duracion_min") > 1,
               F.col("trip_distance") / (F.col("duracion_min") / 60))
         .otherwise(F.lit(None))
    )
    .withColumn(
        "tip_pct",
        F.when(F.col("fare_amount") > 0,
               F.col("tip_amount") / F.col("fare_amount") * 100)
         .otherwise(F.lit(0.0))
    )
    .withColumn(
        "categoria_viaje",
        F.when(F.col("trip_distance") < 1,   "micro")
         .when(F.col("trip_distance") < 3,   "corto")
         .when(F.col("trip_distance") < 10,  "medio")
         .otherwise("largo")
    )
    .filter(F.col("duracion_min").between(1, 180))
)

enriquecido.show(5, truncate=False)
print(f"\nFilas despues de limpieza y enriquecimiento: a calcular con count()")

In [ ]:
# --- groupBy + agg multidimensional ---
# Esta celda dispara un shuffle (Exchange) por el groupBy
metricas = (
    enriquecido
    .groupBy("pickup_hour", "categoria_viaje")
    .agg(
        F.count("*").alias("viajes"),
        F.round(F.avg("fare_amount"),           2).alias("tarifa_prom"),
        F.round(F.avg("tip_pct"),               2).alias("tip_pct_prom"),
        F.round(F.avg("duracion_min"),          1).alias("duracion_prom_min"),
        F.round(F.avg("velocidad_mph"),         1).alias("velocidad_prom_mph"),
        F.round(F.stddev("fare_amount"),        2).alias("tarifa_std"),
        F.round(F.percentile_approx("fare_amount", 0.9), 2).alias("tarifa_p90"),
    )
    .orderBy("pickup_hour", "categoria_viaje")
)

metricas.show(10, truncate=False)
print("\nPlan — observa el Exchange (shuffle) introducido por groupBy:")
metricas.explain("formatted")

In [ ]:
# --- Window functions ---
# rank(), dense_rank(), sum acumulado
w_hora   = Window.partitionBy("pickup_hour").orderBy(F.desc("viajes"))
w_global = Window.orderBy(F.desc("viajes"))
w_acum   = Window.orderBy("pickup_hour").rowsBetween(Window.unboundedPreceding, 0)

top_por_hora = (
    metricas
    .withColumn("rank_en_hora",   F.rank().over(w_hora))
    .withColumn("rank_global",    F.dense_rank().over(w_global))
    .withColumn("viajes_acum",    F.sum("viajes").over(w_acum))
    .filter(F.col("rank_en_hora") <= 2)
    .orderBy("pickup_hour", "rank_en_hora")
)

top_por_hora.select(
    "pickup_hour", "categoria_viaje", "viajes",
    "tarifa_prom", "rank_en_hora", "rank_global", "viajes_acum"
).show(20, truncate=False)

In [ ]:
# --- Join con broadcast ---
# Una tabla de referencia pequena se replica en todos los executors (BroadcastHashJoin)
# Esto evita el shuffle que causaria un SortMergeJoin

zip_ref = (
    sdf_full.select("pickup_zip")
    .where(F.col("pickup_zip").isNotNull())
    .distinct()
    .limit(500)
    .withColumn(
        "borough",
        F.when(F.col("pickup_zip").between(10001, 10099), "Manhattan")
         .when(F.col("pickup_zip").between(11200, 11299), "Brooklyn")
         .when(F.col("pickup_zip").between(11300, 11399), "Queens")
         .when(F.col("pickup_zip").between(10450, 10499), "Bronx")
         .otherwise("Other")
    )
)

t0 = time.perf_counter()
joined = (
    enriquecido
    .join(F.broadcast(zip_ref), on="pickup_zip", how="left")
    .groupBy("borough")
    .agg(
        F.count("*").alias("viajes"),
        F.round(F.avg("fare_amount"), 2).alias("tarifa_prom"),
        F.round(F.avg("tip_pct"),     2).alias("tip_pct_prom"),
    )
    .orderBy(F.desc("viajes"))
)
joined.show(truncate=False)
print(f"Join broadcast + groupBy: {time.perf_counter()-t0:.2f}s")
print("\nBusca 'BroadcastHashJoin' en el plan:")
joined.explain("formatted")

In [ ]:
# --- Pivot ---
# Convierte valores de una columna en columnas separadas
# Es un wide transformation (shuffle interno)
pivot_dow = (
    enriquecido
    .groupBy("pickup_hour")
    .pivot("categoria_viaje", ["micro", "corto", "medio", "largo"])
    .agg(F.count("*"))
    .orderBy("pickup_hour")
)
pivot_dow.show(truncate=False)

In [ ]:
# --- Calidad de datos: nulos, duplicados, fillna ---
print("=== Nulos por columna ===")
sdf_full.select([
    F.round(
        F.sum(F.col(c).isNull().cast("int")) / F.count("*") * 100, 2
    ).alias(c)
    for c in ["fare_amount", "tip_amount", "trip_distance", "pickup_zip", "passenger_count"]
]).show(truncate=False)

# Limpieza
limpio = (
    sdf_full
    .na.drop(subset=["fare_amount", "trip_distance"])
    .na.fill({"tip_amount": 0.0, "passenger_count": 1})
    .filter(F.col("fare_amount") > 0)
    .filter(F.col("trip_distance") > 0)
    .dropDuplicates(["tpep_pickup_datetime", "tpep_dropoff_datetime", "fare_amount"])
)
print(f"Filas despues de limpieza: {limpio.count():,} (de {TOTAL:,})")

In [ ]:
# --- UDF vs pandas_udf vs funcion nativa ---
# Este benchmark muestra el costo de salir del motor nativo de Spark

import pandas as pd
from pyspark.sql.functions import udf, pandas_udf
from pyspark.sql.types import StringType

muestra = enriquecido.limit(300_000)

# 1. Python UDF clasica: fila por fila, serializa Python/JVM en cada fila
@udf(StringType())
def clasifica_udf(minutos):
    '''Clasifica duracion — Python UDF clasica.'''
    if minutos is None: return None
    if minutos < 5:  return "rapido"
    if minutos < 20: return "normal"
    return "largo"

t0 = time.perf_counter()
muestra.withColumn("tipo", clasifica_udf(F.col("duracion_min"))).groupBy("tipo").count().show()
t_udf = time.perf_counter() - t0

# 2. Pandas UDF: vectorizada, opera sobre Series completas
@pandas_udf(StringType())
def clasifica_pandas_udf(s: pd.Series) -> pd.Series:
    '''Clasifica duracion — Pandas UDF vectorizada.'''
    return s.map(lambda x: "rapido" if (x is not None and x < 5) else (
                            "normal" if (x is not None and x < 20) else "largo"))

t0 = time.perf_counter()
muestra.withColumn("tipo", clasifica_pandas_udf(F.col("duracion_min"))).groupBy("tipo").count().show()
t_pandas_udf = time.perf_counter() - t0

# 3. Funcion nativa: Catalyst puede optimizarla, sin cruzar Python/JVM
t0 = time.perf_counter()
muestra.withColumn("tipo",
    F.when(F.col("duracion_min") < 5,  "rapido")
     .when(F.col("duracion_min") < 20, "normal")
     .otherwise("largo")
).groupBy("tipo").count().show()
t_nativa = time.perf_counter() - t0

print(f"Python UDF    : {t_udf:.2f}s")
print(f"Pandas UDF    : {t_pandas_udf:.2f}s")
print(f"Funcion nativa: {t_nativa:.2f}s")
if t_nativa > 0:
    print(f"\nSpeedup nativa vs Python UDF  : {t_udf   /t_nativa:.1f}x")
    print(f"Speedup nativa vs Pandas UDF  : {t_pandas_udf/t_nativa:.1f}x")
print("\nOrden de preferencia: nativa > pandas_udf > udf")

---
# Sección 6 -- Por que Spark sobre Pandas -- y cuando no

## La pregunta correcta

No es "cual es mas rapido". Es "cual minimiza costo, tiempo y riesgo para este problema".

### Pandas: fortalezas reales

- API muy expresiva para exploracion y analisis interactivo.
- Cero overhead distribuido cuando los datos caben en memoria.
- Integracion directa con scikit-learn, matplotlib, seaborn.
- Iteracion rapida: prototipas en minutos.

### Spark: fortalezas reales

- **Plan optimizado por Catalyst**: el codigo Python no es lo que Spark ejecuta — hay un optimizador en el medio que puede reordenar operaciones, empujar filtros, elegir estrategias de join.
- **Escala mas alla de la RAM**: Spark puede procesar datasets mas grandes que la memoria disponible mediante spill a disco controlado.
- **Observabilidad operativa**: Query Profile, Spark UI, metricas de stages y tasks.
- **SQL distribuido nativo**: Spark SQL es un motor SQL completo, no un agregado.
- **Pipeline reproducible**: el mismo notebook se convierte en un job programado sin cambios.
- **Delta Lake**: integrado, con ACID, historial y upserts.

### Lo que Pandas NO puede hacer a escala

Cuando tienes 200 GB de datos y 32 GB de RAM:

```python
# En Pandas: MemoryError o swap que mata el proceso
pdf = pd.read_parquet("s3://datos/enorme.parquet")  # falla

# En Spark: procesa en particiones, spill controlado si necesario
sdf = spark.read.parquet("s3://datos/enorme.parquet")
resultado = sdf.groupBy("ciudad").agg(...)  # funciona
```

In [ ]:
# Comparacion representativa: misma operacion, misma muestra de datos
# La clave: incluir toPandas() en el tiempo de Pandas (eso es parte del costo real)
import time
import pandas as pd
from pyspark.sql import functions as F

MUESTRA_N = 500_000
MUESTRA = spark.read.table("samples.nyctaxi.trips").limit(MUESTRA_N)

# --- Spark: plan lazy + accion completa ---
t0 = time.perf_counter()
resultado_spark = (
    MUESTRA
    .filter(F.col("fare_amount") > 0)
    .withColumn("hora", F.hour("tpep_pickup_datetime"))
    .groupBy("hora")
    .agg(
        F.count("*").alias("viajes"),
        F.round(F.avg("fare_amount"), 2).alias("tarifa_prom"),
        F.round(F.percentile_approx("fare_amount", 0.9), 2).alias("p90"),
    )
    .orderBy("hora")
)
resultado_spark.show(5)
t_spark = time.perf_counter() - t0

# --- Pandas: transferir datos al driver + groupby local ---
t0 = time.perf_counter()
pdf = MUESTRA.select("fare_amount", "tpep_pickup_datetime").toPandas()
pdf = pdf[pdf["fare_amount"] > 0].copy()
pdf["hora"] = pd.to_datetime(pdf["tpep_pickup_datetime"]).dt.hour
pdf_res = (
    pdf.groupby("hora")["fare_amount"]
    .agg(["count", "mean", lambda x: x.quantile(0.9)])
    .rename(columns={"count": "viajes", "mean": "tarifa_prom", "<lambda_0>": "p90"})
    .sort_index()
)
print(pdf_res.head(5))
t_pandas = time.perf_counter() - t0

print(f"\nSpark  (plan + accion completa): {t_spark:.2f}s")
print(f"Pandas (toPandas + groupby)    : {t_pandas:.2f}s")
print()
print("Leccion: toPandas() transfiere datos al driver.")
print("Ese costo de red/serializar es parte del tiempo total de Pandas.")
print("En produccion, si no necesitas los datos en Python local, Spark lo evita.")

## Tabla de decision: Pandas vs Spark

| Criterio | Elige Pandas | Elige Spark |
|---|---|---|
| Tamano del dataset | Cabe en RAM confortablemente | > RAM disponible o crece con el tiempo |
| Velocidad de iteracion | Alta prioridad | Pipeline estable, no exploracion |
| Integracion scikit-learn | Necesaria y directa | Requiere MLlib o bridge |
| Observabilidad del pipeline | No critica | Critica (Query Profile, logs) |
| SQL distribuido | No necesario | Nativo y potente |
| Equipo mixto SQL + Python | — | Ventaja clara |
| Tablas con historial y rollback | No | Delta Lake integrado |
| Dataset que crece 10x en un ano | Riesgo alto | Escala sin reescribir |

### Cuando Pandas sigue siendo la respuesta correcta

- Analisis exploratorio rapido sobre una muestra pequena.
- Integracion directa con visualizacion (matplotlib, seaborn, plotly).
- El dataset tiene 200 MB y no va a crecer.
- El equipo ya domina Pandas y el cuello de botella no es computacional.

---
# Sección 7 -- Por que Spark sobre Dask -- y cuando no

## Diferencias arquitecturales profundas

A primera vista, Dask y Spark parecen similares: ambos son lazy, ambos particionan datos,
ambos permiten paralelismo. Pero la arquitectura interna es fundamentalmente diferente.

### Dask: toda la pila es Python

```
+----------------------+
|   Tu codigo Python   |
|   dask.dataframe     |
+----------+-----------+
           |
    Task Graph (Python)
           |
    +------+-------+
    | Dask Scheduler|  <- Python
    +------+-------+
    |      |       |
  Worker  Worker  Worker  <- Procesos Python
```

Ventaja: integracion natural con numpy, scipy, scikit-learn.
La serializacion es Python a Python — sin cruzar la JVM.

### Spark: plan en JVM, cliente Python

```
+----------------------+
|   Tu codigo Python   |
|   pyspark / Spark    |
|   Connect            |
+----------+-----------+
           | gRPC / Py4J
    +------+-------+
    |  Spark Server |  <- JVM
    |  Catalyst     |
    |  Tungsten      |
    +------+-------+
    |      |       |
  Executor Executor Executor  <- JVM threads
```

**La diferencia clave**: Catalyst actua **antes** de ejecutar cualquier tarea.
Puede reordenar filtros, eliminar columnas innecesarias, elegir entre BroadcastHashJoin
y SortMergeJoin, y fusionar operaciones. Dask ejecuta el grafo de tareas Python tal como
se definio, sin ese nivel de optimizacion automatica.

In [ ]:
# Benchmark Dask vs Spark: mismas operaciones, misma fuente de datos
import time
import dask.dataframe as dd
import pandas as pd
from pyspark.sql import functions as F

# Dataset comun materializado en memoria para comparacion justa
MUESTRA_N = 400_000
print(f"Materializando {MUESTRA_N:,} filas para comparacion...")
t0 = time.perf_counter()
pdf_base = spark.read.table("samples.nyctaxi.trips").limit(MUESTRA_N).toPandas()
pdf_base = pdf_base[pdf_base["fare_amount"] > 0].copy()
print(f"Materializacion: {time.perf_counter()-t0:.2f}s | filas={len(pdf_base):,}")

# Dask DataFrame desde pandas
ddf = dd.from_pandas(pdf_base, npartitions=8)
# Spark DataFrame desde pandas
sdf_bench = spark.createDataFrame(pdf_base)

print("\n=== Operacion 1: groupBy hora + promedio + percentil ===")

t0 = time.perf_counter()
dask_res = (
    ddf.assign(hora=dd.to_datetime(ddf["tpep_pickup_datetime"]).dt.hour)
       .groupby("hora")["fare_amount"]
       .agg(["count", "mean"])
       .compute()
       .sort_index()
)
t_dask = time.perf_counter() - t0
print(dask_res.head(3))

t0 = time.perf_counter()
spark_res = (
    sdf_bench
    .withColumn("hora", F.hour("tpep_pickup_datetime"))
    .groupBy("hora")
    .agg(
        F.count("*").alias("count"),
        F.round(F.avg("fare_amount"), 2).alias("mean"),
        F.round(F.percentile_approx("fare_amount", 0.9), 2).alias("p90"),
    )
    .orderBy("hora")
)
spark_res.show(3)
t_spark = time.perf_counter() - t0

print(f"Dask  groupBy + compute: {t_dask:.3f}s")
print(f"Spark groupBy + show:   {t_spark:.3f}s")

In [ ]:
# Operacion 2: join con tabla de referencia
print("=== Operacion 2: join con tabla de referencia ===")

zips_unicos = pdf_base["pickup_zip"].dropna().unique()[:100]
ref_pdf = pd.DataFrame({"pickup_zip": zips_unicos, "zona": "referencia"})
ref_ddf = dd.from_pandas(ref_pdf, npartitions=1)
ref_sdf = spark.createDataFrame(ref_pdf)

t0 = time.perf_counter()
dask_join = ddf.merge(ref_ddf, on="pickup_zip", how="inner").compute()
t_dask_join = time.perf_counter() - t0

t0 = time.perf_counter()
spark_join = sdf_bench.join(F.broadcast(ref_sdf), on="pickup_zip", how="inner")
n_spark = spark_join.count()
t_spark_join = time.perf_counter() - t0

print(f"Dask  join + compute: {t_dask_join:.3f}s | filas={len(dask_join):,}")
print(f"Spark join + count:   {t_spark_join:.3f}s | filas={n_spark:,}")

print("\nPlan del join Spark — busca BroadcastHashJoin:")
spark_join.explain("formatted")

## Tabla de decision: Dask vs Spark

| Criterio | Elige Dask | Elige Spark |
|---|---|---|
| Ecosistema scipy/numpy/sklearn | Natural, sin JVM overhead | Requiere bridge (pandas_udf, MLlib) |
| Migracion desde Pandas | Gradual, API muy similar | Cambio de mentalidad mas grande |
| Optimizacion automatica de joins | Limitada | Robusta (Catalyst, estadisticas) |
| Predicate pushdown | Parcial | Profundo (nivel de archivo Delta) |
| SQL distribuido nativo | Limitado | Completo |
| Ecosistema de produccion | Emergente, creciendo | Maduro, estandar industrial |
| Observabilidad | Dask Dashboard (bokeh) | Query Profile, Spark UI |
| Plataforma gestionada | Coiled, Saturn Cloud | Databricks, EMR, Dataproc |
| Tablas Delta / ACID | No nativo | Integrado |
| Streaming | Dask Streams (limitado) | Structured Streaming (maduro) |

### Cuando Dask es la respuesta correcta

- El pipeline ya usa numpy/scipy y necesitas escalar sin reescribir logica de matrices.
- El equipo conoce bien Pandas y quiere escalar con la menor friccion posible.
- El problema es mas de data science que de data engineering.
- No necesitas ACID, historial de datos ni join muy complejos.

### Cuando Spark gana claramente

- Necesitas joins entre tablas grandes con observabilidad del plan.
- El pipeline tiene que funcionar en produccion de forma reproducible y monitoreable.
- El equipo incluye analistas SQL que necesitan trabajar sobre los mismos datos.
- Los datos crecen y necesitas una plataforma que escale sin rediseno.

---
# Sección 8 -- Delta Lake avanzado: MERGE, Time Travel, VACUUM

## Por que Delta Lake y no solo Parquet

Un lago de datos con archivos Parquet sueltos no ofrece, por si mismo:
- Transacciones atomicas (si el job falla a la mitad, los datos quedan en estado inconsistente).
- Historial de versiones (no sabes que cambio ni cuando).
- Rollback (si llegan datos malos, no puedes volver atras facilmente).
- Upserts (tienes que reescribir particiones completas).

Delta Lake resuelve todos esos problemas con un **transaction log** que registra cada operacion.

### Arquitectura del transaction log

```
mi_tabla/
  _delta_log/
    00000000000000000000.json   <- version 0: CREATE TABLE
    00000000000000000001.json   <- version 1: primera escritura
    00000000000000000002.json   <- version 2: OPTIMIZE
    00000000000000000003.json   <- version 3: MERGE
    00000000000000000010.checkpoint.parquet  <- checkpoint cada N versiones
  part-00000-xxx.parquet
  part-00001-xxx.parquet
  ...
```

Cada archivo JSON del log lista que archivos se agregaron y cuales se marcaron como eliminados.
Los archivos fisicos no se borran hasta que se ejecuta `VACUUM`.

In [ ]:
# Crear tabla Delta de trabajo para esta seccion
from pyspark.sql import functions as F

CATALOG = spark.sql("SELECT current_catalog()").first()[0]
SCHEMA  = spark.sql("SELECT current_schema()").first()[0]
DELTA_MAIN = f"{CATALOG}.{SCHEMA}.taxi_sesion9_main"

print(f"Creando tabla Delta: {DELTA_MAIN}")

spark.sql(f'''
    CREATE OR REPLACE TABLE {DELTA_MAIN}
    CLUSTER BY (tpep_pickup_datetime, pickup_zip)
    AS
    SELECT
        CAST(
            row_number() OVER (ORDER BY tpep_pickup_datetime)
            AS BIGINT
        ) AS trip_id,
        tpep_pickup_datetime,
        tpep_dropoff_datetime,
        pickup_zip,
        dropoff_zip,
        fare_amount,
        tip_amount,
        trip_distance,
        CAST(1 AS INT) AS es_valido
    FROM samples.nyctaxi.trips
    WHERE fare_amount > 0
      AND trip_distance > 0
''')

n = spark.read.table(DELTA_MAIN).count()
print(f"Tabla creada: {n:,} filas")
spark.sql(f"DESCRIBE DETAIL {DELTA_MAIN}").select(
    "format", "clusteringColumns", "numFiles", "sizeInBytes"
).show(truncate=False)

In [ ]:
# MERGE (upsert): el patron mas poderoso de Delta Lake
# Sirve para: actualizaciones parciales, inserciones de filas nuevas,
# eliminacion logica de registros invalidos

from delta.tables import DeltaTable

# Preparar filas de actualizacion y nuevas
actualizaciones = spark.createDataFrame([
    (1,   0),   # trip_id=1 -> marcar como invalido
    (2,   0),   # trip_id=2 -> marcar como invalido
    (5,   0),   # trip_id=5 -> marcar como invalido
    (999999997, 1),  # fila nueva sintetica
    (999999998, 1),  # fila nueva sintetica
], ["trip_id", "es_valido"])

target = DeltaTable.forName(spark, DELTA_MAIN)

(
    target.alias("t")
    .merge(actualizaciones.alias("s"), "t.trip_id = s.trip_id")
    .whenMatchedUpdate(set={"es_valido": "s.es_valido"})
    .whenNotMatchedInsert(values={
        "trip_id":              "s.trip_id",
        "tpep_pickup_datetime": "CAST(NULL AS TIMESTAMP)",
        "tpep_dropoff_datetime":"CAST(NULL AS TIMESTAMP)",
        "pickup_zip":           "CAST(NULL AS INT)",
        "dropoff_zip":          "CAST(NULL AS INT)",
        "fare_amount":          "CAST(0 AS DOUBLE)",
        "tip_amount":           "CAST(0 AS DOUBLE)",
        "trip_distance":        "CAST(0 AS DOUBLE)",
        "es_valido":            "s.es_valido",
    })
    .execute()
)

print("MERGE ejecutado. Historial de la tabla:")
spark.sql(f"DESCRIBE HISTORY {DELTA_MAIN}").select(
    "version", "timestamp", "operation", "operationMetrics"
).show(5, truncate=False)

In [ ]:
# Time Travel: leer una version anterior de la tabla
print("=== Time Travel ===")

# Por numero de version
print("Version 0 (antes del MERGE):")
v0_count = (
    spark.read.format("delta")
         .option("versionAsOf", 0)
         .table(DELTA_MAIN)
         .count()
)
print(f"  Filas en version 0: {v0_count:,}")

v_actual = spark.read.table(DELTA_MAIN).count()
print(f"  Filas en version actual: {v_actual:,}")

# Restaurar a version anterior
print("\nRestaurando a version 0 (antes del MERGE)...")
spark.sql(f"RESTORE TABLE {DELTA_MAIN} TO VERSION AS OF 0")

spark.sql(f"DESCRIBE HISTORY {DELTA_MAIN}").select(
    "version", "timestamp", "operation"
).show(5, truncate=False)

In [ ]:
# VACUUM: eliminar archivos fisicos obsoletos
# En produccion: RETAIN 7 DAYS (el default)
# En laboratorio: RETAIN 0 HOURS con override para ver el efecto

print("Antes de VACUUM:")
spark.sql(f"DESCRIBE DETAIL {DELTA_MAIN}").select("numFiles", "sizeInBytes").show()

# Deshabilitar la verificacion de retencion minima (solo para laboratorio)
spark.conf.set("spark.databricks.delta.retentionDurationCheck.enabled", "false")
spark.sql(f"VACUUM {DELTA_MAIN} RETAIN 0 HOURS")
spark.conf.set("spark.databricks.delta.retentionDurationCheck.enabled", "true")

print("\nDespues de VACUUM:")
spark.sql(f"DESCRIBE DETAIL {DELTA_MAIN}").select("numFiles", "sizeInBytes").show()
print("Nota: despues de VACUUM con 0 horas ya no es posible hacer time travel a versiones eliminadas.")

---
# Sección 9 -- Delta Live Tables (DLT): el patron medallion

## Que es Delta Live Tables

**DLT** es el sistema de pipelines declarativos de Databricks.
En lugar de escribir notebooks imperatives ("primero haz esto, luego aquello"),
con DLT describes **que tablas existen y de donde vienen**. Databricks resuelve el orden
de ejecucion, maneja dependencias y aplica actualizaciones incrementales automaticamente.

### Comparacion: notebook interactivo vs DLT

| Aspecto | Notebook interactivo | DLT Pipeline |
|---|---|---|
| Ejecucion | Manual, celda a celda | Orquestada por Databricks |
| Dependencias | Implicitas (orden de celdas) | Explicitas (nombres de tablas) |
| Calidad del dato | Manual (`assert`, `raise`) | Declarativa (`@dlt.expect`) |
| Actualizacion incremental | Manual | Automatica (STREAMING LIVE TABLE) |
| Observabilidad | Query Profile | DLT Event Log + lineage |
| Reintentos | Manuales | Automaticos |

### El patron Medallion

El estandar de arquitectura de datos en lakehouse:

```
[Fuente raw]
     |
     v
[Bronze]  <- ingesta sin transformar, fidelidad maxima al origen
     |
     v
[Silver]  <- limpieza, enriquecimiento, validacion de calidad
     |
     v
[Gold]    <- agregaciones listas para dashboards, ML, reportes
```

In [ ]:
# IMPORTANTE: el codigo de DLT no se puede ejecutar en un notebook interactivo.
# Esta celda IMPRIME el patron de codigo — ejecutarlo en un DLT Pipeline.
#
# Para usar DLT:
# 1. Ir a Databricks -> Workflows -> Delta Live Tables
# 2. Crear un nuevo pipeline
# 3. Apuntar a un notebook que contenga el codigo siguiente
# 4. Configurar el target catalog y schema
# 5. Ejecutar "Full refresh"

PIPELINE_CODE = '''
import dlt
from pyspark.sql import functions as F

# ---- CAPA BRONZE: ingesta raw ----
@dlt.table(
    name="taxi_bronze",
    comment="Datos crudos de NYC Taxi desde Unity Catalog",
    table_properties={"quality": "bronze"}
)
def taxi_bronze():
    return spark.read.table("samples.nyctaxi.trips")

# ---- CAPA SILVER: limpieza y enriquecimiento ----
@dlt.table(
    name="taxi_silver",
    comment="Datos limpios con columnas derivadas",
    table_properties={"quality": "silver"}
)
@dlt.expect_or_drop("fare_positivo",        "fare_amount > 0")
@dlt.expect_or_drop("distancia_positiva",   "trip_distance > 0")
@dlt.expect("duracion_razonable",           "duracion_min BETWEEN 1 AND 180")
def taxi_silver():
    return (
        dlt.read("taxi_bronze")
        .filter(F.col("fare_amount").between(1, 200))
        .withColumn("pickup_hour", F.hour("tpep_pickup_datetime"))
        .withColumn(
            "duracion_min",
            (F.unix_timestamp("tpep_dropoff_datetime")
             - F.unix_timestamp("tpep_pickup_datetime")) / 60
        )
        .withColumn(
            "tip_pct",
            F.when(F.col("fare_amount") > 0,
                   F.col("tip_amount") / F.col("fare_amount") * 100)
             .otherwise(F.lit(0.0))
        )
        .withColumn(
            "categoria_viaje",
            F.when(F.col("trip_distance") < 1,   "micro")
             .when(F.col("trip_distance") < 3,   "corto")
             .when(F.col("trip_distance") < 10,  "medio")
             .otherwise("largo")
        )
    )

# ---- CAPA GOLD: agregaciones para analitica ----
@dlt.table(
    name="taxi_gold_hourly",
    comment="Metricas por hora del dia, listas para dashboards",
    table_properties={"quality": "gold"}
)
def taxi_gold_hourly():
    return (
        dlt.read("taxi_silver")
        .groupBy("pickup_hour", "categoria_viaje")
        .agg(
            F.count("*").alias("viajes"),
            F.round(F.avg("fare_amount"), 2).alias("tarifa_prom"),
            F.round(F.avg("tip_pct"),     2).alias("tip_pct_prom"),
            F.round(F.percentile_approx("fare_amount", 0.9), 2).alias("tarifa_p90"),
        )
        .orderBy("pickup_hour", "categoria_viaje")
    )
'''

print(PIPELINE_CODE)
print("=" * 60)
print("Para ejecutar este pipeline en Databricks:")
print("  1. Workflows -> Delta Live Tables -> Create pipeline")
print("  2. Source: apuntar a un notebook con el codigo anterior")
print("  3. Target catalog y schema: donde se crearan las tablas")
print("  4. Clic en 'Full refresh'")

## Ventajas operativas de DLT sobre notebooks manuales

### Calidad del dato declarativa

Con `@dlt.expect_or_drop("nombre_regla", "condicion SQL")`, Databricks:
- Evalua la condicion en cada fila.
- Si la condicion falla, **descarta la fila** de la tabla destino.
- Registra cuantas filas fallaron en el Event Log.

Sin DLT, tendrias que escribir esos controles manualmente y ninguna infraestructura los monitorea.

### Actualizaciones incrementales

Con `STREAMING LIVE TABLE`, DLT puede procesar **solo los datos nuevos** desde la ultima ejecucion.
Esto reduce enormemente el costo y el tiempo de pipelines que se ejecutan cada hora o cada dia.

### Lineage automatico

Databricks registra que tablas dependen de cuales. Si cambias `taxi_bronze`, el lineage
muestra automaticamente que `taxi_silver` y `taxi_gold_hourly` seran afectadas.

---
# Sección 10 -- Taller: pipeline end-to-end serverless

## Ejercicios del taller

Los tres ejercicios siguientes estan disenados para Databricks Serverless con Unity Catalog.
Cada celda lanza un `NotImplementedError` si no completas la solucion.

### Objetivos del taller

- Aplicar Window functions en un analisis real.
- Ejecutar un MERGE en una tabla Delta propia.
- Construir un reporte de calidad de datos programatico.

In [ ]:
# Ejercicio 1 — Window Functions y ranking
# ─────────────────────────────────────────
# Con el dataset samples.nyctaxi.trips, construye un DataFrame que muestre:
#   - Para cada hora del dia (pickup_hour)
#   - Las 3 categorias de viaje (micro / corto / medio / largo) con mayor tip_pct promedio
#   - Filtra grupos con menos de 100 viajes (no son estadisticamente representativos)
#   - Columnas esperadas: [pickup_hour, categoria_viaje, tip_pct_prom, viajes, rank]
#
# Pista: usa Window.partitionBy("pickup_hour").orderBy(F.desc("tip_pct_prom"))
#        y F.rank().over(w) o F.dense_rank().over(w)

raise NotImplementedError(
    "Completa este ejercicio.\n"
    "1. Carga samples.nyctaxi.trips y crea tip_pct y categoria_viaje.\n"
    "2. groupBy pickup_hour + categoria_viaje, calcula tip_pct_prom y viajes.\n"
    "3. Filtra grupos con viajes >= 100.\n"
    "4. Aplica rank() dentro de cada hora.\n"
    "5. Filtra rank <= 3 y muestra el resultado."
)

In [ ]:
# Ejercicio 2 — MERGE en Delta
# ────────────────────────────
# a) Crea una tabla Delta en tu schema con los viajes del pickup_zip mas frecuente
#    del dataset samples.nyctaxi.trips.
#    La tabla debe tener al menos: trip_id (row_number), pickup_zip, fare_amount, es_valido (=1)
#
# b) Usa DeltaTable.forName().merge() para:
#    - Marcar es_valido=0 en todos los viajes donde fare_amount > 100
#    - Insertar 3 filas nuevas con trip_id sintetico > 900000
#
# c) Muestra DESCRIBE HISTORY de la tabla para verificar el MERGE.

raise NotImplementedError(
    "Completa este ejercicio.\n"
    "1. Encuentra el pickup_zip mas frecuente con groupBy + orderBy + limit(1).\n"
    "2. Crea la tabla Delta con CREATE OR REPLACE TABLE ... AS SELECT ...\n"
    "3. Construye el DataFrame de actualizaciones (trip_id, es_valido).\n"
    "4. Ejecuta el MERGE con whenMatchedUpdate y whenNotMatchedInsert.\n"
    "5. Verifica con DESCRIBE HISTORY."
)

In [ ]:
# Ejercicio 3 — Reporte de calidad de datos
# ─────────────────────────────────────────
# Construye un DataFrame con columnas [metrica, valor] que incluya:
#
# Para cada columna en ["fare_amount", "tip_amount", "trip_distance", "pickup_zip"]:
#   - "pct_nulos_{col}": porcentaje de valores nulos
#   - "pct_negativos_{col}": porcentaje de valores < 0 (para columnas numericas)
#
# Ademas:
#   - "top5_pickup_zip": los 5 pickup_zip mas frecuentes como string "zip:count, ..."
#   - "total_filas": numero total de filas del dataset
#
# El resultado debe ser un DataFrame con exactamente 2 columnas: metrica y valor (string).
#
# Pista: usa F.sum(F.col(c).isNull().cast("int")) / F.count("*") * 100 para pct nulos.
#        Para construir el reporte usa spark.createDataFrame([("nombre", "valor"), ...])

raise NotImplementedError(
    "Completa este ejercicio.\n"
    "1. Calcula pct_nulos para cada columna relevante.\n"
    "2. Calcula pct_negativos para columnas numericas.\n"
    "3. Calcula top5_pickup_zip con groupBy + orderBy + limit.\n"
    "4. Une todo en un DataFrame de 2 columnas: [metrica, valor].\n"
    "5. Muestra el resultado ordenado por metrica."
)

---
## Checklist final: Databricks Serverless 2025

Antes de cerrar el notebook, verifica que puedes responder estas preguntas con evidencia:

```
[ ] Uso catalog.schema.table (Unity Catalog) en lugar de rutas /dbfs/
[ ] Evite sparkContext y APIs RDD que fallan en Spark Connect
[ ] Instale dependencias con %pip (no %sh pip)
[ ] Use saveAsTable() para persistir — no save() con rutas legacy
[ ] Defini columnas de Liquid Clustering relevantes a mis queries
[ ] Conozco la diferencia entre OPTIMIZE, VACUUM y Predictive Optimization
[ ] Puedo escribir un pipeline DLT basico con bronze, silver y gold
[ ] Priorizo funciones nativas > pandas_udf > udf en ese orden
[ ] Se por que Spark Connect cambia que APIs estan disponibles
[ ] Tengo criterios claros para elegir Pandas, Dask o Spark segun el problema
```

Si puedes marcar todos los items con evidencia del notebook, ya no estas "ejecutando celdas":
estas **razonando sobre la plataforma de datos moderna**.